In [ ]:
import os, time, glob
from pathlib import Path
import pandas as pd
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

KM_ID = 'ckm00285@naver.com'
KM_PW = 'ckm620700!'
SM_ID = 'cth00284@naver.com'
SM_PW = 'cth620700!'

DOWNLOAD_DIR = str(Path.home() / "Downloads")


def wait_download_complete(download_dir, pattern="supplier_order_list_*.xlsx", since_ts=None, need_count=1, timeout=120):
    """since_ts 이후에 생성/수정된 supplier_order_list_*.xlsx 파일을 need_count개 찾을 때까지 대기"""
    end = time.time() + timeout
    while time.time() < end:
        # 다운로드 중 파일이 있으면 대기

        if glob.glob(os.path.join(download_dir, "*.crdownload")):
            time.sleep(0.3)
            continue

        files = glob.glob(os.path.join(download_dir, pattern))
        if since_ts is not None:
            files = [f for f in files if os.path.getmtime(f) >= since_ts]
        files.sort(key=os.path.getmtime, reverse=True)

        if len(files) >= need_count:
            return files[:need_count]

        time.sleep(0.3)

    raise TimeoutError("다운로드된 supplier_order_list 파일을 시간 내에 찾지 못했어요.")


def merge_supplier_excels(file1, file2, out_path):
    """file1의 헤더(컬럼명/순서)를 기준으로 고정하고 file2는 헤더 제외 데이터만 이어붙여 저장"""
    df1 = pd.read_excel(file1, dtype=str)
    df2 = pd.read_excel(file2, dtype=str)

    cols = list(df1.columns)
    df2 = df2.reindex(columns=cols)

    merged = pd.concat([df1, df2], ignore_index=True)
    merged.to_excel(out_path, index=False)


# ✅ 새롭게 붙인 코드 (계정 1개 다운로드 작업을 함수로 묶음)
def download_onchannel_excel(account_id, account_pw, account_name="ACCOUNT"):
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 15)

    try:
        # 로그인 페이지로 진입 (너는 이미 로그인 화면에서 시작하는 경우가 많지만, 안정성 위해 추가)
        driver.get('https://www.onch3.co.kr/login/login_web.php')

        # ID 입력
        ID = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body > div.container > form > div:nth-child(2) > input')))
        ID.clear()
        ID.send_keys(account_id)

        # PW 입력
        PW = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body > div.container > form > div:nth-child(3) > input')))
        PW.clear()
        PW.send_keys(account_pw)

        # 로그인 버튼
        login_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'body > div.container > form > button')))
        login_btn.click()

        # 관리자 페이지 이동
        driver.get('https://www.onch3.co.kr/mypage/dashboard.php?target=seller')

        # 공급사 정보 클릭
        seller_cate = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_sidebar_menu > div:nth-child(8) > a > span.menu-title.text-gray-600')))
        seller_cate.click()

        # 주문 정보 클릭
        oredr_info = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_sidebar_menu > div.menu-item.menu-accordion.hover.show > div > div:nth-child(5) > a > span')))
        oredr_info.click()

        # 주문내역 다운로드 버튼
        order_down_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_app_wrapper > div.contentWrap > div.container.p-3 > div.contentWrap > div.card > div.card-header.min-h-auto.p-6.border-bottom-0.d-flex.justify-content-between.align-items-center > div > button.btn.btn-excel.me-2')))
        order_down_btn.click()

        # 기간 설정
        today = datetime.today()
        start_day = today - timedelta(days=7)
        start_str = start_day.strftime("%Y-%m-%d")
        end_str   = today.strftime("%Y-%m-%d")
        start_sel = "#downExcelOrderListModal > div > div > div.modal-body.p-4 > div.row.align-items-center.mb-4 > div.col > div > input:nth-child(1)"
        end_sel   = "#downExcelOrderListModal > div > div > div.modal-body.p-4 > div.row.align-items-center.mb-4 > div.col > div > input:nth-child(3)"

        wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#downExcelOrderListModal")))

        def send_date(selector: str, value: str):
            el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))

            driver.execute_script("""
                arguments[0].removeAttribute('readonly');
                arguments[0].removeAttribute('disabled');
            """, el)

            # 클릭 가로채기 방지 (JS로 focus/click)
            driver.execute_script("arguments[0].focus(); arguments[0].click();", el)

            el.send_keys(Keys.CONTROL, 'a')
            el.send_keys(Keys.DELETE)
            el.send_keys(value)
            el.send_keys(Keys.TAB)

        send_date(start_sel, start_str)
        send_date(end_sel, end_str)
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)

        # (있으면) alert 처리
        try:
            alert = WebDriverWait(driver, 2).until(EC.alert_is_present())
            print(f"[{account_name}] alert 문구:", alert.text)
            alert.accept()
        except:
            pass

        # 동의 체크
        agree = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#agreement-order-down-check")))
        if not agree.is_selected():
            driver.execute_script("arguments[0].click();", agree)

        # ✅ 새롭게 붙인 코드 (다운로드 클릭 직전에 시간 기록)
        t0 = time.time()

        # 다운로드 클릭
        down_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#btn-order-excel-down")))
        driver.execute_script("arguments[0].click();", down_btn)  # ✅ 새롭게 붙인 코드 (JS 클릭)
        

        # (있으면) alert 처리
        try:
            alert = WebDriverWait(driver, 3).until(EC.alert_is_present())
            print(f"[{account_name}] alert 문구:", alert.text)
            alert.accept()
        except:
            pass

        # ✅ 새롭게 붙인 코드 (방금 다운로드된 파일 1개 경로 확보)
        downloaded_file = wait_download_complete(DOWNLOAD_DIR, since_ts=t0, need_count=1, timeout=120)[0]
        print(f"✅ [{account_name}] 다운로드 파일:", downloaded_file)

        # # 모달 닫기
        # close_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#downExcelOrderListModal > div > div > div.modal-footer.justify-content-center > button.btn.btn-dark.px-4')))
        # driver.execute_script("arguments[0].click();", close_btn)

        return downloaded_file

    finally:
    # ✅ 어떤 경우에도 반드시 종료
        try:
            driver.quit()
        except:
            pass

# =========================================================
# ✅ 새롭게 붙인 코드 (SM → KM → 합쳐 저장)
# =========================================================

# 여기 변수는 네가 이미 가지고 있는 걸 그대로 사용:
# SM_ID, SM_PW, KM_ID, KM_PW

sm_file = download_onchannel_excel(SM_ID, SM_PW, account_name="SM")
km_file = download_onchannel_excel(KM_ID, KM_PW, account_name="KM")

out_path = os.path.join(DOWNLOAD_DIR, "온채널 수집파일.xlsx")
try:
    merge_supplier_excels(sm_file, km_file, out_path)
except PermissionError:
    out_path = os.path.join(DOWNLOAD_DIR, f"온채널 수집파일_{time.strftime('%Y%m%d_%H%M%S')}.xlsx")
    merge_supplier_excels(sm_file, km_file, out_path)

print("✅ 합친 파일 저장 완료:", out_path)

[SM] alert 문구: 다운로드가 완료되었습니다.
✅ [SM] 다운로드 파일: C:\Users\ckm00\Downloads\supplier_order_list_1772172432722.xlsx
[KM] alert 문구: 다운로드가 완료되었습니다.
✅ [KM] 다운로드 파일: C:\Users\ckm00\Downloads\supplier_order_list_1772172448722.xlsx
✅ 합친 파일 저장 완료: C:\Users\ckm00\Downloads\온채널 주문내역.xlsx


In [5]:
import os, time, glob
from pathlib import Path
import pandas as pd
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

KM_ID = 'ckm00285@naver.com'
KM_PW = 'ckm620700!'
SM_ID = 'cth00284@naver.com'
SM_PW = 'cth620700!'

DOWNLOAD_DIR = str(Path.home() / "Downloads")


def wait_download_complete(download_dir, pattern="supplier_order_list_*.xlsx", since_ts=None, need_count=1, timeout=120):
    """since_ts 이후에 생성/수정된 pattern 파일을 need_count개 찾을 때까지 대기"""
    end = time.time() + timeout

    while time.time() < end:
        # ✅ since_ts 이후에 만들어진 .crdownload만 '다운로드 진행중'으로 간주
        crs = glob.glob(os.path.join(download_dir, "*.crdownload"))
        if since_ts is not None:
            crs = [c for c in crs if os.path.getmtime(c) >= since_ts]
        if crs:
            time.sleep(0.3)
            continue

        files = glob.glob(os.path.join(download_dir, pattern))
        if since_ts is not None:
            files = [f for f in files if os.path.getmtime(f) >= since_ts]
        files.sort(key=os.path.getmtime, reverse=True)

        if len(files) >= need_count:
            return files[:need_count]

        time.sleep(0.3)

    raise TimeoutError(f"다운로드된 {pattern} 파일을 시간 내에 찾지 못했어요.")


def merge_supplier_excels(file1, file2, out_path):
    """file1의 헤더(컬럼명/순서)를 기준으로 고정하고 file2는 헤더 제외 데이터만 이어붙여 저장"""
    df1 = pd.read_excel(file1, dtype=str)
    df2 = pd.read_excel(file2, dtype=str)

    cols = list(df1.columns)
    df2 = df2.reindex(columns=cols)

    merged = pd.concat([df1, df2], ignore_index=True)
    merged.to_excel(out_path, index=False)


# ✅ 새롭게 붙인 코드 (계정 1개 다운로드 작업을 함수로 묶음)
def download_onchannel_excel(account_id, account_pw, account_name="ACCOUNT"):
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 15)

    try:
        # 로그인 페이지로 진입 (너는 이미 로그인 화면에서 시작하는 경우가 많지만, 안정성 위해 추가)
        driver.get('https://www.onch3.co.kr/login/login_web.php')

        # ID 입력
        ID = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body > div.container > form > div:nth-child(2) > input')))
        ID.clear()
        ID.send_keys(account_id)

        # PW 입력
        PW = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body > div.container > form > div:nth-child(3) > input')))
        PW.clear()
        PW.send_keys(account_pw)

        # 로그인 버튼
        login_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'body > div.container > form > button')))
        login_btn.click()

        # 관리자 페이지 이동
        driver.get('https://www.onch3.co.kr/mypage/dashboard.php?target=seller')

        # 공급사 정보 클릭
        seller_cate = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_sidebar_menu > div:nth-child(8) > a > span.menu-title.text-gray-600')))
        seller_cate.click()

        # 주문 정보 클릭
        oredr_info = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_sidebar_menu > div.menu-item.menu-accordion.hover.show > div > div:nth-child(5) > a > span')))
        oredr_info.click()

        # 주문내역 다운로드 버튼
        order_down_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_app_wrapper > div.contentWrap > div.container.p-3 > div.contentWrap > div.card > div.card-header.min-h-auto.p-6.border-bottom-0.d-flex.justify-content-between.align-items-center > div > button.btn.btn-excel.me-2')))
        order_down_btn.click()

        # 기간 설정
        today = datetime.today()
        start_day = today - timedelta(days=7)
        start_str = start_day.strftime("%Y-%m-%d")
        end_str   = today.strftime("%Y-%m-%d")
        start_sel = "#downExcelOrderListModal > div > div > div.modal-body.p-4 > div.row.align-items-center.mb-4 > div.col > div > input:nth-child(1)"
        end_sel   = "#downExcelOrderListModal > div > div > div.modal-body.p-4 > div.row.align-items-center.mb-4 > div.col > div > input:nth-child(3)"

        wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#downExcelOrderListModal")))

        def send_date(selector: str, value: str):
            el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))

            driver.execute_script("""
                arguments[0].removeAttribute('readonly');
                arguments[0].removeAttribute('disabled');
            """, el)

            # 클릭 가로채기 방지 (JS로 focus/click)
            driver.execute_script("arguments[0].focus(); arguments[0].click();", el)

            el.send_keys(Keys.CONTROL, 'a')
            el.send_keys(Keys.DELETE)
            el.send_keys(value)
            el.send_keys(Keys.TAB)

        send_date(start_sel, start_str)
        send_date(end_sel, end_str)
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)

        # (있으면) alert 처리
        try:
            alert = WebDriverWait(driver, 2).until(EC.alert_is_present())
            print(f"[{account_name}] alert 문구:", alert.text)
            alert.accept()
        except:
            pass

        # 동의 체크
        agree = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#agreement-order-down-check")))
        if not agree.is_selected():
            driver.execute_script("arguments[0].click();", agree)

        # ✅ 새롭게 붙인 코드 (다운로드 클릭 직전에 시간 기록)
        t0 = time.time()

        # 다운로드 클릭
        down_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#btn-order-excel-down")))
        driver.execute_script("arguments[0].click();", down_btn)  # ✅ 새롭게 붙인 코드 (JS 클릭)
        

        # (있으면) alert 처리
        try:
            alert = WebDriverWait(driver, 3).until(EC.alert_is_present())
            print(f"[{account_name}] alert 문구:", alert.text)
            alert.accept()
        except:
            pass

        # ✅ 새롭게 붙인 코드 (방금 다운로드된 파일 1개 경로 확보)
        downloaded_file = wait_download_complete(DOWNLOAD_DIR, since_ts=t0, need_count=1, timeout=120)[0]
        print(f"✅ [{account_name}] 다운로드 파일:", downloaded_file)

        # # 모달 닫기
        # close_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#downExcelOrderListModal > div > div > div.modal-footer.justify-content-center > button.btn.btn-dark.px-4')))
        # driver.execute_script("arguments[0].click();", close_btn)

        return downloaded_file

    finally:
    # ✅ 어떤 경우에도 반드시 종료
        try:
            driver.quit()
        except:
            pass

# =========================================================
# ✅ 새롭게 붙인 코드 (SM → KM → 합쳐 저장)
# =========================================================

# 여기 변수는 네가 이미 가지고 있는 걸 그대로 사용:
# SM_ID, SM_PW, KM_ID, KM_PW

sm_file = download_onchannel_excel(SM_ID, SM_PW, account_name="SM")
km_file = download_onchannel_excel(KM_ID, KM_PW, account_name="KM")

out_path = os.path.join(DOWNLOAD_DIR, "온채널 주문내역.xlsx")
try:
    merge_supplier_excels(sm_file, km_file, out_path)
except PermissionError:
    out_path = os.path.join(DOWNLOAD_DIR, f"온채널 주문내역_{time.strftime('%Y%m%d_%H%M%S')}.xlsx")
    merge_supplier_excels(sm_file, km_file, out_path)

print("✅ 합친 파일 저장 완료:", out_path)

[SM] alert 문구: 다운로드가 완료되었습니다.
✅ [SM] 다운로드 파일: C:\Users\ckm00\Downloads\supplier_order_list_1772160730376.xlsx
[KM] alert 문구: 다운로드가 완료되었습니다.
✅ [KM] 다운로드 파일: C:\Users\ckm00\Downloads\supplier_order_list_1772160745948.xlsx
✅ 합친 파일 저장 완료: C:\Users\ckm00\Downloads\온채널 주문내역_20260227_115232.xlsx


In [ ]:
import os, time, glob
from pathlib import Path
import pandas as pd
from datetime import datetime, timedelta

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options


KM_ID = 'ckm00285@naver.com'
KM_PW = 'ckm620700!'
SM_ID = 'cth00284@naver.com'
SM_PW = 'cth620700!'

DOWNLOAD_DIR = str(Path.home() / "Downloads")


def make_driver(download_dir: str):
    options = Options()

    # ✅ 크롬 창 안 뜨게 (headless)
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    # ✅ 안정성 옵션
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=ko-KR")

    # ✅ 다운로드 경로 고정 + 다운로드 팝업 차단
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)

    # ✅ headless에서 다운로드 막히는 환경 대비 (CDP로 강제 허용)
    try:
        driver.execute_cdp_cmd(
            "Page.setDownloadBehavior",
            {"behavior": "allow", "downloadPath": download_dir}
        )
    except Exception:
        pass

    return driver


def wait_download_complete(download_dir, pattern="supplier_order_list_*.xlsx", since_ts=None, need_count=1, timeout=120):
    """since_ts 이후에 생성/수정된 pattern 파일을 need_count개 찾을 때까지 대기"""
    end = time.time() + timeout

    while time.time() < end:
        # ✅ since_ts 이후에 만들어진 .crdownload만 '다운로드 진행중'으로 간주
        crs = glob.glob(os.path.join(download_dir, "*.crdownload"))
        if since_ts is not None:
            crs = [c for c in crs if os.path.getmtime(c) >= since_ts]
        if crs:
            time.sleep(0.3)
            continue

        files = glob.glob(os.path.join(download_dir, pattern))
        if since_ts is not None:
            files = [f for f in files if os.path.getmtime(f) >= since_ts]
        files.sort(key=os.path.getmtime, reverse=True)

        if len(files) >= need_count:
            return files[:need_count]

        time.sleep(0.3)

    raise TimeoutError(f"다운로드된 {pattern} 파일을 시간 내에 찾지 못했어요.")


def merge_supplier_excels(file1, file2, out_path):
    """file1의 헤더(컬럼명/순서)를 기준으로 고정하고 file2는 헤더 제외 데이터만 이어붙여 저장"""
    df1 = pd.read_excel(file1, dtype=str)
    df2 = pd.read_excel(file2, dtype=str)

    cols = list(df1.columns)
    df2 = df2.reindex(columns=cols)

    merged = pd.concat([df1, df2], ignore_index=True)
    merged.to_excel(out_path, index=False)


def download_onchannel_excel(account_id, account_pw, account_name="ACCOUNT"):
    # ✅ 여기서 headless 드라이버 사용
    driver = make_driver(DOWNLOAD_DIR)
    wait = WebDriverWait(driver, 15)

    try:
        # 로그인 페이지
        driver.get('https://www.onch3.co.kr/login/login_web.php')

        # ID 입력
        ID = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body > div.container > form > div:nth-child(2) > input')))
        ID.clear()
        ID.send_keys(account_id)

        # PW 입력
        PW = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'body > div.container > form > div:nth-child(3) > input')))
        PW.clear()
        PW.send_keys(account_pw)

        # 로그인 버튼
        login_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'body > div.container > form > button')))
        login_btn.click()

        # 관리자 페이지 이동
        driver.get('https://www.onch3.co.kr/mypage/dashboard.php?target=seller')

        # 공급사 정보 클릭
        seller_cate = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_sidebar_menu > div:nth-child(8) > a > span.menu-title.text-gray-600')))
        seller_cate.click()

        # 주문 정보 클릭
        oredr_info = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_sidebar_menu > div.menu-item.menu-accordion.hover.show > div > div:nth-child(5) > a > span')))
        oredr_info.click()

        # 주문내역 다운로드 버튼
        order_down_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '#kt_app_wrapper > div.contentWrap > div.container.p-3 > div.contentWrap > div.card > div.card-header.min-h-auto.p-6.border-bottom-0.d-flex.justify-content-between.align-items-center > div > button.btn.btn-excel.me-2')))
        order_down_btn.click()

        # 기간 설정
        today = datetime.today()
        start_day = today - timedelta(days=7)
        start_str = start_day.strftime("%Y-%m-%d")
        end_str   = today.strftime("%Y-%m-%d")
        start_sel = "#downExcelOrderListModal > div > div > div.modal-body.p-4 > div.row.align-items-center.mb-4 > div.col > div > input:nth-child(1)"
        end_sel   = "#downExcelOrderListModal > div > div > div.modal-body.p-4 > div.row.align-items-center.mb-4 > div.col > div > input:nth-child(3)"

        wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#downExcelOrderListModal")))

        def send_date(selector: str, value: str):
            el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))

            driver.execute_script("""
                arguments[0].removeAttribute('readonly');
                arguments[0].removeAttribute('disabled');
            """, el)

            driver.execute_script("arguments[0].focus(); arguments[0].click();", el)

            el.send_keys(Keys.CONTROL, 'a')
            el.send_keys(Keys.DELETE)
            el.send_keys(value)
            el.send_keys(Keys.TAB)

        send_date(start_sel, start_str)
        send_date(end_sel, end_str)
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)

        # (있으면) alert 처리
        try:
            alert = WebDriverWait(driver, 2).until(EC.alert_is_present())
            print(f"[{account_name}] alert 문구:", alert.text)
            alert.accept()
        except Exception:
            pass

        # 동의 체크
        agree = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#agreement-order-down-check")))
        if not agree.is_selected():
            driver.execute_script("arguments[0].click();", agree)

        # ✅ 다운로드 클릭 직전에 시간 기록
        t0 = time.time()

        # 다운로드 클릭
        down_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#btn-order-excel-down")))
        driver.execute_script("arguments[0].click();", down_btn)

        # (있으면) alert 처리
        try:
            alert = WebDriverWait(driver, 3).until(EC.alert_is_present())
            print(f"[{account_name}] alert 문구:", alert.text)
            alert.accept()
        except Exception:
            pass

        # ✅ 방금 다운로드된 파일 1개 경로 확보
        downloaded_file = wait_download_complete(DOWNLOAD_DIR, since_ts=t0, need_count=1, timeout=120)[0]
        print(f"✅ [{account_name}] 다운로드 파일:", downloaded_file)

        return downloaded_file

    finally:
        # ✅ 어떤 경우에도 반드시 종료
        try:
            driver.quit()
        except Exception:
            pass


# =========================================================
# ✅ SM → KM → 합쳐 저장
# =========================================================
sm_file = download_onchannel_excel(SM_ID, SM_PW, account_name="SM")
km_file = download_onchannel_excel(KM_ID, KM_PW, account_name="KM")

out_path = os.path.join(DOWNLOAD_DIR, "온채널 주문내역.xlsx")
try:
    merge_supplier_excels(sm_file, km_file, out_path)
except PermissionError:
    out_path = os.path.join(DOWNLOAD_DIR, f"온채널 주문내역_{time.strftime('%Y%m%d_%H%M%S')}.xlsx")
    merge_supplier_excels(sm_file, km_file, out_path)

print("✅ 합친 파일 저장 완료:", out_path)





[SM] alert 문구: 다운로드가 완료되었습니다.
✅ [SM] 다운로드 파일: C:\Users\ckm00\Downloads\supplier_order_list_1772173078951.xlsx
[KM] alert 문구: 다운로드가 완료되었습니다.
✅ [KM] 다운로드 파일: C:\Users\ckm00\Downloads\supplier_order_list_1772173094727.xlsx
✅ 합친 파일 저장 완료: C:\Users\ckm00\Downloads\온채널 주문내역.xlsx
